# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 tabular dataset using the `mlcroissant` library.

### Dataset Source
This dataset is accessible via a Croissant schema JSON-LD file at:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

The dataset describes clinicopathological and molecular variables of 77 cancer survivors with second primary colorectal cancer, including MSI-H status, anatomical distribution, demographics, comorbidities, and other clinical variables.

In [ ]:
# Install mlcroissant to enable Croissant dataset access
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Display basic information
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Size: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

FAIR^2 dataset is structured with record sets, fields, and columns, referenced by their `@id`. Let's enumerate what's available.

In [ ]:
# Explore available record sets in the dataset
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name} | @id: {rs.id}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name}")
    print("\n")
# For demonstration, show first 2 records from each record set
for rs in record_sets:
    print(f"-- Sample records from RecordSet @{rs.id} --")
    records = list(dataset.records(record_set=rs.id))
    print(records[:2])
    print("\n")

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for easy analysis.
Each DataFrame corresponds to a particular record set (by its `@id`), and columns correspond to field `@id`s.

In [ ]:
# Collect all record set @id values for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load each record set into a DataFrame, keeping columns as field @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @{record_set_id}: Columns = {df.columns.tolist()} | Shape = {df.shape}")

# Show first few rows for first available record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print("\nSample from first record set:")
    print(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering, normalization, and grouping.

**All entity references use their `@id`.**

Let's:
- Pick a numeric field (e.g., age, interval between diagnoses, etc.)
- Filter cases by age > 60 (if age field exists)
- Normalize the age within filtered cases
- Group filtered records by another field (e.g., MSI/MMR status or anatomical location)

Fields referenced by their Croissant `@id`.

In [ ]:
# Identify a numeric field and a grouping field for analysis
# Demo names: replace with actual field @id after inspection from previous step
record_set_id = record_set_ids[0]  # use the first record set
df = dataframes[record_set_id]

# Find numeric and grouping field @id from field listing
numeric_field_id = None
group_field_id = None
fields = [f.id for f in record_sets[0].fields]
field_names = [f.name for f in record_sets[0].fields]
# Attempt to pick 'age' and 'sex' or similar
for f in record_sets[0].fields:
    if 'age' in f.name.lower():
        numeric_field_id = f.id
    if 'sex' in f.name.lower() or 'msi' in f.name.lower():
        group_field_id = f.id
    if numeric_field_id and group_field_id:
        break

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Filter records and normalize numeric_field
if numeric_field_id and numeric_field_id in df.columns:
    threshold = 60
    try:
        # Convert field to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception:
        pass
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric_field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped (average) {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("Numeric field not found or unavailable in this record set.")

## 5. Visualization
Visualize data distributions and relationships using the extracted DataFrames.

Example: Histogram of a numeric field (e.g., age), boxplot by categorical (e.g., MSI/MMR status).

All fields referenced by their `@id`.

In [ ]:
# Plot histogram & boxplot of numeric field
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].dropna().astype(float).hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id} (e.g., Age)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if numeric_field_id and group_field_id and numeric_field_id in df.columns and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.suptitle("")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore the FAIR^2 dataset using the `mlcroissant` library. All record sets, fields, and columns were referenced via their `@id`, ensuring robust and reproducible access.

Key findings and validations:
- The dataset contains detailed clinical, molecular, and anatomical information for second primary CRC in cancer survivors.
- Record sets, fields, and columns can be programmatically explored and visualized based on their Croissant `@id`.
- Exploratory analysis enables filtering, normalization, and grouping by clinical variables of interest.

For further research, this workflow ensures extensible and FAIR access for downstream analytics, clinical research, and model validation using Croissant standardized schemas.